In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("seeds/seeds_dataset.txt", sep=r"\s+", header=None)

df.columns = [
    "area", "perimeter", "compactness",
    "kernel_length", "kernel_width",
    "asymmetry_coefficient", "groove_length",
    "class"
]

In [5]:
print(df.head(5))

    area  perimeter  compactness  kernel_length  kernel_width  \
0  15.26      14.84       0.8710          5.763         3.312   
1  14.88      14.57       0.8811          5.554         3.333   
2  14.29      14.09       0.9050          5.291         3.337   
3  13.84      13.94       0.8955          5.324         3.379   
4  16.14      14.99       0.9034          5.658         3.562   

   asymmetry_coefficient  groove_length  class  
0                  2.221          5.220      1  
1                  1.018          4.956      1  
2                  2.699          4.825      1  
3                  2.259          4.805      1  
4                  1.355          5.175      1  


A1: chuẩn hóa từng đặc trưng về mean 0 std 1, bỏ cột class

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
).drop(columns=["class"])

In [7]:
print(df_scaled.head(5))

       area  perimeter  compactness  kernel_length  kernel_width  \
0  0.142098   0.215462     0.000061       0.304218      0.141702   
1  0.011188   0.008224     0.428515      -0.168625      0.197432   
2 -0.192067  -0.360201     1.442383      -0.763637      0.208048   
3 -0.347091  -0.475333     1.039381      -0.688978      0.319508   
4  0.445257   0.330595     1.374509       0.066666      0.805159   

   asymmetry_coefficient  groove_length  
0              -0.986152      -0.383577  
1              -1.788166      -0.922013  
2              -0.667479      -1.189192  
3              -0.960818      -1.229983  
4              -1.563495      -0.475356  


A2: Tính tích khoảng cách euclid của 5 mẫu đầu tiên. d(A,B)= sqrt(dot(A-B,A-B))

In [8]:
X = df_scaled.iloc[:5].to_numpy()
d_euclidean=np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        d_vector=X[i,:]-X[j,:]
        d_euclidean[i,j]=np.dot(d_vector,d_vector)**0.5

d_euclidean_df = pd.DataFrame(d_euclidean, index=range(1, 6), columns=range(1, 6))
print(d_euclidean_df)

          1         2         3         4         5
1  0.000000  1.184688  2.102054  1.879306  1.683041
2  1.184688  0.000000  1.698957  1.341778  1.364521
3  2.102054  1.698957  0.000000  0.552637  1.801760
4  1.879306  1.341778  0.552637  0.000000  1.768949
5  1.683041  1.364521  1.801760  1.768949  0.000000


A3: tính tích khoảng cách có trọng số của 5 mẫu đầu tiên

In [29]:
A = np.diag(np.array([1, 1, 0, 1, 1, 0, 0]))
d_matrix_A=np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        d_vector=X[i,:]-X[j,:]
        d_matrix_A[i,j]=np.dot(d_vector,A @ d_vector)**0.5

d_matrix_A = pd.DataFrame(d_matrix_A, index=range(1, 6), columns=range(1, 6))
print(d_matrix_A)


          1         2         3         4         5
1  0.000000  0.535510  1.260068  1.317026  0.775737
2  0.535510  0.000000  0.728836  0.804898  0.846778
3  1.260068  0.728836  0.000000  0.235129  1.389002
4  1.317026  0.804898  0.235129  0.000000  1.443672
5  0.775737  0.846778  1.389002  1.443672  0.000000


A4: so sánh mẫu gần nhau nhất ở hai cách tính khoảng cách

In [31]:
print('Euclidean distance')
d_euclidean_df_copy = d_euclidean_df.copy()
d_euclidean_df_copy+= np.diag([float('inf')]*5)
for i in range(5):
    print(f'{i+1} is nearest to : { d_euclidean_df_copy.iloc[i].idxmin()}')

Euclidean distance
1 is nearest to : 2
2 is nearest to : 1
3 is nearest to : 4
4 is nearest to : 3
5 is nearest to : 2


In [30]:
print('Dot product with weighted matrix')
d_matrix_A_copy = d_matrix_A.copy()
d_matrix_A_copy+= np.diag([float('inf')]*5)
for i in range(5):
    print(f'{i+1} is nearest to : { d_matrix_A_copy.iloc[i].idxmin()}')

Dot product with weighted matrix
1 is nearest to : 2
2 is nearest to : 1
3 is nearest to : 4
4 is nearest to : 3
5 is nearest to : 1


=> Thứ tự thay đổi nhỏ ở mẫu thứ 5. Lí do vì trong phép tính khoảng cách lần 2, các đặc trưng kích thước được gán trọng số 1, các đặc trưng còn lại trọng số 0. Vậy nên kết quả có nghĩa là xét về tổng thể thì mẫu 5 giống mẫu 2 nhất, nhưng xét riêng về kích thước thì mẫu 5 giống mẫu 1 hơn.  

A5: Lấy hai mẫu đầu làm cơ sở, trực chuẩn cơ sở, 